# 3. 진단 검정

추정한 식이 가정을 지키는지 검정합니다: 잔차 자기상관·이분산·정규성·함수 형태·구조 변화, 계수 제약, 변수 더하기·빼기.
이어서 시계열의 단위근과 공적분을 보고, 공적분 관계로 오차수정모형을 만듭니다. 검정은 모두 같은 결과 객체(`TestResult`)를 돌려줍니다.

In [ ]:
import equion as eqn

wf = eqn.datasets.load("toy_q")
eq = wf.equation("eq_i").ls("I c Y(-1) R")          # 참 모형과 같은 꼴의 투자식

## 3.1 표준 진단 한 표

In [ ]:
eqn.show(eq.diagnostics())

p값이 작으면(보통 0.05 미만) 귀무가설을 기각합니다 — 예를 들어 첫 줄이 작으면 잔차에 계열상관이 있다는 뜻입니다.
이 식은 참 모형과 같은 꼴이라 대부분 기각되지 않습니다. ARCH만 5%에서 기각됐는데, 참 모형의 오차는 분산이 일정하므로
우연입니다 — 5% 검정은 맞는 모형에서도 스무 번에 한 번꼴로 기각합니다. 검정 여러 개를 함께 볼 때 기억해 둘 점입니다.

## 3.2 검정 결과 다루기

검정 하나하나는 메서드로 부르고(부르면 요약 한 줄이 찍힌다), 결과는 통계량·p값·귀무가설·세부 수치를 가진 객체입니다.

In [ ]:
lm = eq.serial_lm(4)                    # Breusch-Godfrey LM, 시차 4까지
print(round(lm.statistic, 4), round(lm.pvalue, 4), lm.null)
print(lm.reject(0.05))                  # 5%에서 기각하는가
eqn.show(lm)                            # 표로 보기

## 3.3 잔차 — 자기상관·이분산·정규성·함수 형태

In [ ]:
q = eq.ljungbox(8)                      # Ljung-Box Q(시차별 표는 q.table)
eq.white()                              # White 이분산(교차항 포함, cross=False면 빼고)
eq.breusch_pagan()                      # Breusch-Pagan-Godfrey
eq.arch(4)                              # ARCH LM — 분산의 자기상관
eq.normality()                          # Jarque-Bera
eq.reset(2)                             # Ramsey RESET — 적합값 제곱·세제곱을 더하면 나아지는가
print(q.table.round(3).head(3))

잘못 쓴 식에서는 검정이 문제를 잡아냅니다. 소비를 정부지출만으로 설명하면 빠진 소득의 움직임이 잔차에 남아 강하게
자기상관하고(DW가 2보다 훨씬 작다), 함수 형태 검정도 기각됩니다.

In [ ]:
bad = wf.equation("bad").ls("CN c G")
bad.serial_lm(4)
bad.reset(1)
print(round(bad.stats["durbin_watson"], 3))

## 3.4 구조 변화와 안정성

정책금리 `R`의 평균은 1995년 무렵 크게 낮아지지만, 투자식의 계수 자체는 변하지 않았습니다(참 모형). Chow 검정은
그 시점 앞뒤의 계수가 같은지, CUSUM은 재귀 잔차의 누적합이 경계를 벗어나는지 봅니다.

In [ ]:
eq.chow("1995Q1")
cs = eq.cusum()                         # 통계량 = 누적합이 경계에 가장 가까이 간 비율(1보다 작으면 경계 안 — 안정)
rc = eq.recursive()                     # 재귀 계수: 처음 몇 기로 시작해 한 기씩 늘려 추정
print(rc.iloc[[20, 100, -1]].round(3))

## 3.5 계수 제약

`c(n)`은 식에 쓴 항의 순서입니다(상수항 `c`가 `c(1)`). 참값 `Y(-1)` 0.15, `R` −1.0을 함께 검정해 봅니다.

In [ ]:
eq.wald("c(2) = 0.15", "c(3) = -1")
eq.wald("c(2) = 0.2")                   # 틀린 값은 기각된다

## 3.6 변수를 더하거나 빼도 되는가

In [ ]:
eq.omitted("R(-1)", "G")                # 더하면 설명력이 유의하게 느는가(참 모형에는 없다)
eq.redundant("R")                       # 빼도 되는가(참 모형에 있다 — 기각)

## 3.7 단위근

수준 시계열(소득 `Y`)은 추세를 따라 움직여 단위근을 가지는 경우가 많습니다. ADF는 "단위근이 있다", KPSS는 "정상이다"가
귀무가설이라 서로 반대 방향입니다. 시리즈 이름 대신 식을 줄 수 있습니다.

In [ ]:
wf.adf("log(Y)", trend="ct")            # 상수 + 추세 — 단위근을 기각하지 못한다
wf.adf("dlog(Y)")                       # 차분하면 정상(기각)
wf.kpss("dlog(Y)")                      # 정상을 기각하지 못한다

## 3.8 공적분

장기금리 `RL`과 정책금리 `R`은 각자 단위근을 가지지만 `RL − R`은 정상입니다(공적분, 참 모형 `RL = 1 + R + u`).

In [ ]:
wf.coint("RL", "R")                     # Engle-Granger: 장기 회귀 잔차의 단위근 검정
eqn.show(wf.johansen("log(M2) log(P) log(Y) RL"))     # Johansen: 여러 변수의 공적분 순위

## 3.9 오차수정모형

공적분 관계가 있으면 단기 변화는 균형에서 벗어난 만큼 되돌아오는 꼴로 씁니다:
`d(RL) = α·(RL − a − b·R)(-1) + … `. `ecm`이 장기식과 단기식을 한 번에 추정합니다.

In [ ]:
ecm = wf.ecm("ecm_rl", "RL", ["R"])
eqn.show(ecm)
print(round(ecm.adjustment["계수"], 3), round(ecm.half_life, 2))

조정 속도 α가 음수이고 1보다 작으면 균형으로 돌아갑니다(참 모형 `u = 0.7·u(-1) + ε`이므로 α ≈ −0.3, 반감기 약 2기).
단기식 `ecm_rl`은 보통 방정식 객체라 [연립 모델](../../docs/tutorial/04-model.md)에 그대로 넣을 수 있습니다. 장기 계수의 표준오차가 필요하면
`method="nls"`로 한 번에 추정합니다.

## 정리

| 하려는 일 | 코드 |
|---|---|
| 한 표 | `eq.diagnostics()` |
| 잔차 | `.serial_lm(p)` · `.ljungbox(p)` · `.white()` · `.breusch_pagan()` · `.arch(p)` · `.normality()` · `.reset(k)` |
| 안정성 | `.chow("1995Q1")` · `.cusum()` · `.recursive()` |
| 계수·변수 | `.wald("c(2) = 0.5")` · `.omitted("X")` · `.redundant("X")` |
| 시계열 | `wf.adf` · `wf.kpss` · `wf.coint` · `wf.johansen` · `wf.ecm` |
| 결과 객체 | `.statistic` · `.pvalue` · `.null` · `.reject(0.05)` · `eqn.show(…)` |

다음: [4. 연립 모델](../../docs/tutorial/04-model.md). 관련 레시피: [진단 레시피](../../docs/howto/diagnostics.md).